# Enhanced Sentence-BERT Winner Model

This notebook compares contestant occupational profile text with the category and clue text on each Jeopardy! board. It computes preregistered similarity and within-game advantage features, then fits a regularized multinomial logistic regression.

The primary contestant text is `optional_soc_profile_text`, falling back to `occupation_text_for_model`. High-value clues are ordinary J!/DJ! clues whose value is at least 0.8 of that game's maximum value in the same round. Final Jeopardy is not included in that feature.

Only board-visible text and occupation/profile text are predictors. Scores, wagers, answers, winner IDs, names, historical winnings, and postgame metadata are excluded. The held-out test games are read from `Output/test_game_ids.csv`; baseline fitting is deferred.

In [4]:
from pathlib import Path
import html
import re

import numpy as np
import pandas as pd
from IPython.display import display

working_dir = Path.cwd().resolve()
project_candidates = [working_dir, *working_dir.parents]
project_candidates += [working_dir / "Project_1", *(parent / "Project_1" for parent in working_dir.parents)]
PROJECT_DIR = next(
    (
        candidate for candidate in project_candidates
        if (candidate / "Data" / "final_contestant_games.csv").exists()
        and (candidate / "Output" / "test_game_ids.csv").exists()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        f"Could not locate Project_1 from the notebook working directory: {working_dir}"
    )
DATA_DIR = PROJECT_DIR / "Data"
OUTPUT_DIR = PROJECT_DIR / "Output"

RANDOM_STATE = 20260925
HIGH_VALUE_RATIO = 0.80
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

FINAL_PATH = DATA_DIR / "final_contestant_games.csv"
RAW_CLUE_PATH = DATA_DIR / "jeopardy_raw_contestant_clue_data.csv.gz"
TEST_GAME_IDS_PATH = OUTPUT_DIR / "test_game_ids.csv"

final_columns = [
    "game_id", "contestant_id", "contestant_position", "is_winner",
    "occupation_text_for_model", "optional_soc_profile_text",
]
raw_columns = [
    "game_id", "clue_source_record_id", "round", "category",
    "clue_text", "clue_value",
]

contestants = pd.read_csv(
    FINAL_PATH,
    skipinitialspace=True,
    usecols=lambda column: column.strip() in final_columns,
)
contestants.columns = contestants.columns.str.strip()
test_game_ids = set(pd.read_csv(TEST_GAME_IDS_PATH)["game_id"])
known_game_ids = set(contestants["game_id"].unique())
missing_test_game_ids = test_game_ids - known_game_ids
assert not missing_test_game_ids, f"Test game IDs missing from contestant data: {sorted(missing_test_game_ids)[:10]}"
raw_clues = pd.read_csv(RAW_CLUE_PATH, usecols=raw_columns, compression="gzip")
raw_clues = raw_clues[raw_clues["game_id"].isin(contestants["game_id"].unique())].copy()

print(f"Contestant-game rows: {len(contestants):,}")
print(f"Games: {contestants['game_id'].nunique():,}")
print(f"Raw clue rows after game filter: {len(raw_clues):,}")

Contestant-game rows: 11,646
Games: 3,882
Raw clue rows after game filter: 676,164


In [5]:
def clean_text(value):
    value = "" if pd.isna(value) else str(value)
    value = html.unescape(re.sub(r"<[^>]+>", " ", value))
    return re.sub(r"\s+", " ", value).strip()

raw_clues["category"] = raw_clues["category"].map(clean_text)
raw_clues["clue_text"] = raw_clues["clue_text"].map(clean_text)
raw_clues["round"] = raw_clues["round"].astype(str).str.strip().str.upper()
raw_clues["clue_value"] = pd.to_numeric(raw_clues["clue_value"], errors="coerce")

clues = raw_clues.drop_duplicates(
    subset=["game_id", "clue_source_record_id"], keep="first"
).copy()
clues = clues[(clues["category"] != "") & (clues["clue_text"] != "")].copy()

ordinary_rounds = {"J!", "DJ!", "1", "2"}
clues["is_ordinary"] = clues["round"].isin(ordinary_rounds) & clues["clue_value"].notna()
round_max = clues.loc[clues["is_ordinary"]].groupby(
    ["game_id", "round"], as_index=False
)["clue_value"].max().rename(columns={"clue_value": "round_max_value"})
clues = clues.merge(round_max, on=["game_id", "round"], how="left")
clues["relative_value"] = clues["clue_value"] / clues["round_max_value"]
clues["high_value_clue"] = (
    clues["is_ordinary"] & (clues["relative_value"] >= HIGH_VALUE_RATIO)
)

profile = contestants["optional_soc_profile_text"].fillna("").map(clean_text)
occupation = contestants["occupation_text_for_model"].fillna("").map(clean_text)
contestants["profile_text"] = profile.where(profile != "", occupation)

assert len(contestants) == 11_646
assert contestants["game_id"].nunique() == 3_882
assert contestants.groupby("game_id").size().eq(3).all()
assert contestants.groupby("game_id")["is_winner"].sum().eq(1).all()
assert not clues.duplicated(["game_id", "clue_source_record_id"]).any()
assert contestants["profile_text"].ne("").all()

categories = clues[["game_id", "category"]].drop_duplicates().copy()
print(f"Distinct board clues: {len(clues):,}")
print(f"Distinct game-category pairs: {len(categories):,}")
print(f"High-value ordinary clues: {int(clues['high_value_clue'].sum()):,}")
print(f"Games without high-value clues: {clues.groupby('game_id')['high_value_clue'].sum().eq(0).sum():,}")

Distinct board clues: 225,388
Distinct game-category pairs: 50,200
High-value ordinary clues: 87,315
Games without high-value clues: 0


In [6]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    get_ipython().run_line_magic("pip", "install -q sentence-transformers scikit-learn")
    from sentence_transformers import SentenceTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

encoder = SentenceTransformer(EMBEDDING_MODEL_NAME)

unique_texts = pd.Index(
    pd.concat([
        contestants["profile_text"],
        clues["clue_text"],
        categories["category"],
    ], ignore_index=True).unique()
)
embeddings = encoder.encode(
    unique_texts.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
).astype("float32")
text_to_embedding = {
    text: embeddings[index] for index, text in enumerate(unique_texts)
}

print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Unique texts embedded: {len(unique_texts):,}")
print(f"Embedding dimension: {embeddings.shape[1]}")

/opt/anaconda3/envs/my_jupyter_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 4046/4046 [03:25<00:00, 19.69it/s]


Embedding model: all-MiniLM-L6-v2
Unique texts embedded: 258,932
Embedding dimension: 384


In [7]:
feature_rows = []
for game_id, game_contestants in contestants.groupby("game_id", sort=False):
    game_clues = clues[clues["game_id"] == game_id]
    clue_texts = game_clues["clue_text"].tolist()
    category_texts = game_clues["category"].drop_duplicates().tolist()
    high_value_texts = game_clues.loc[
        game_clues["high_value_clue"], "clue_text"
    ].tolist()

    clue_vectors = np.vstack([text_to_embedding[text] for text in clue_texts])
    category_vectors = np.vstack([text_to_embedding[text] for text in category_texts])
    high_value_vectors = (
        np.vstack([text_to_embedding[text] for text in high_value_texts])
        if high_value_texts else None
    )

    for row in game_contestants.itertuples(index=False):
        profile_vector = text_to_embedding[row.profile_text]
        clue_similarities = clue_vectors @ profile_vector
        category_similarities = category_vectors @ profile_vector
        high_value_similarities = (
            high_value_vectors @ profile_vector
            if high_value_vectors is not None else np.array([np.nan])
        )
        feature_rows.append({
            "game_id": row.game_id,
            "contestant_id": row.contestant_id,
            "contestant_position": row.contestant_position,
            "is_winner": row.is_winner,
            "avg_similarity": float(clue_similarities.mean()),
            "max_similarity": float(clue_similarities.max()),
            "category_similarity": float(category_similarities.mean()),
            "high_value_similarity": float(np.nanmean(high_value_similarities)),
            "high_value_similarity_available": int(high_value_vectors is not None),
        })

features = pd.DataFrame(feature_rows)
base_feature_names = [
    "avg_similarity", "max_similarity", "category_similarity",
    "high_value_similarity",
]
for feature_name in base_feature_names:
    other_mean = features.groupby("game_id")[feature_name].transform("sum")
    other_mean = (other_mean - features[feature_name]) / 2.0
    features[f"{feature_name}_advantage"] = features[feature_name] - other_mean

advantage_feature_names = [f"{name}_advantage" for name in base_feature_names]
model_feature_names = base_feature_names + advantage_feature_names

assert len(features) == len(contestants)
assert features[model_feature_names].notna().all().all()
assert np.isfinite(features[model_feature_names].to_numpy()).all()
assert np.allclose(
    features.groupby("game_id")[advantage_feature_names].sum().to_numpy(),
    0.0,
    atol=1e-6,
)

print(f"Model rows: {len(features):,}")
display(features[model_feature_names].describe().T.round(4))

Model rows: 11,646


,count,mean,std,min,25%,50%,75%,max
avg_similarity,11646.0,0.0053,0.0155,-0.0496,-0.0056,0.0042,0.0150,0.0777
max_similarity,11646.0,0.2001,0.0660,0.0545,0.1521,0.1902,0.2381,0.5976
category_similarity,11646.0,0.0539,0.0423,-0.0619,0.0224,0.0530,0.0843,0.2242
high_value_similarity,11646.0,0.0061,0.0186,-0.0582,-0.0067,0.0051,0.0180,0.0979
avg_similarity_advantage,11646.0,0.0000,0.0167,-0.0598,-0.0110,-0.0002,0.0107,0.0651
max_similarity_advantage,11646.0,0.0000,0.0754,-0.3029,-0.0502,-0.0036,0.0456,0.3897
category_similarity_advantage,11646.0,0.0000,0.0465,-0.1635,-0.0322,0.0001,0.0323,0.1718
high_value_similarity_advantage,11646.0,0.0000,0.0198,-0.0775,-0.0131,-0.0001,0.0127,0.0888


In [8]:
available_game_ids = set(features["game_id"].unique())
assert test_game_ids <= available_game_ids, (
    f"Test game IDs missing from model features: {sorted(test_game_ids - available_game_ids)[:10]}"
)
train_game_ids = available_game_ids - test_game_ids
assert train_game_ids.isdisjoint(test_game_ids)

features_by_position = features.sort_values(
    ["game_id", "contestant_position"]
).copy()
wide = features_by_position.pivot(
    index="game_id", columns="contestant_position", values=model_feature_names
)
wide.columns = [f"{feature}_position_{position}" for feature, position in wide.columns]
wide = wide.reset_index()

winner_positions = features.loc[features["is_winner"].eq(1)].set_index(
    "game_id"
)["contestant_position"]
wide["winner_position"] = wide["game_id"].map(winner_positions).astype(int)
wide["game_id"] = wide["game_id"].astype(features["game_id"].dtype)

wide_feature_names = [
    column for column in wide.columns
    if column not in {"game_id", "winner_position"}
]
X = wide[wide_feature_names]
y = wide["winner_position"]
groups = wide["game_id"]
train_mask = groups.isin(train_game_ids).to_numpy()
test_mask = groups.isin(test_game_ids).to_numpy()

model_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        solver="lbfgs",
        max_iter=2_000,
        random_state=RANDOM_STATE,
    )),
])

regularization_grid = {"logistic__C": np.logspace(-3, 3, 7)}
inner_cv = GroupKFold(n_splits=5)
search = GridSearchCV(
    model_pipeline,
    regularization_grid,
    scoring="neg_log_loss",
    cv=inner_cv,
    n_jobs=-1,
    refit=True,
)
search.fit(X.loc[train_mask], y.loc[train_mask], groups=groups.loc[train_mask])

heldout_probabilities = search.predict_proba(X.loc[test_mask])
heldout_predictions = search.predict(X.loc[test_mask])
class_order = search.best_estimator_.named_steps["logistic"].classes_

assert np.array_equal(class_order, np.array([1, 2, 3]))
assert np.allclose(heldout_probabilities.sum(axis=1), 1.0, atol=1e-8)

print(f"Training games: {train_mask.sum():,}")
print(f"Held-out games: {test_mask.sum():,}")
print(f"Best C: {search.best_params_['logistic__C']}")

Training games: 3,106
Held-out games: 776
Best C: 0.001


In [9]:
test_games = wide.loc[test_mask, ["game_id", "winner_position"]].reset_index(drop=True)
predictions = test_games.copy()
for index, position in enumerate(class_order):
    predictions[f"probability_position_{position}"] = heldout_probabilities[:, index]
predictions["predicted_winner_position"] = heldout_predictions
predictions["probability_sum"] = predictions[
    [f"probability_position_{position}" for position in class_order]
].sum(axis=1)

uniform_probabilities = np.full_like(heldout_probabilities, 1 / 3)
metrics = pd.DataFrame([
    {
        "model": "enhanced_sentence_bert",
        "log_loss": log_loss(y.loc[test_mask], heldout_probabilities, labels=class_order),
        "accuracy": accuracy_score(y.loc[test_mask], heldout_predictions),
        "games": int(test_mask.sum()),
        "best_C": float(search.best_params_["logistic__C"]),
    },
    {
        "model": "uniform_reference",
        "log_loss": log_loss(y.loc[test_mask], uniform_probabilities, labels=class_order),
        "accuracy": np.nan,
        "games": int(test_mask.sum()),
        "best_C": np.nan,
    },
])

split_ids = pd.DataFrame({
    "game_id": wide["game_id"],
    "split": np.where(train_mask, "train", "test"),
})
feature_summary = features[model_feature_names].describe().T.reset_index()
feature_summary = feature_summary.rename(columns={"index": "feature"})

predictions.to_csv(OUTPUT_DIR / "enhanced_model_test_predictions.csv", index=False)
metrics.to_csv(OUTPUT_DIR / "enhanced_model_metrics.csv", index=False)
features.to_csv(OUTPUT_DIR / "enhanced_model_features.csv", index=False)
feature_summary.to_csv(OUTPUT_DIR / "enhanced_model_feature_summary.csv", index=False)
split_ids.to_csv(OUTPUT_DIR / "enhanced_model_game_splits.csv", index=False)

assert set(predictions["game_id"]) == test_game_ids
assert np.allclose(predictions["probability_sum"], 1.0, atol=1e-8)
assert not set(split_ids.loc[split_ids["split"] == "train", "game_id"]).intersection(
    set(split_ids.loc[split_ids["split"] == "test", "game_id"])
)

display(metrics.round(4))
print(f"Saved model artifacts to {OUTPUT_DIR}")

,model,log_loss,accuracy,games,best_C
0,enhanced_sentence_bert,1.0930,0.3557,776,0.001
1,uniform_reference,1.0986,NaN,776,NaN


Saved model artifacts to /Users/Bhargav/Code/DS4002-SLMB/Project_1/Output
